# Ensemble analysis: parameter effects & Sobol sensitivity

Runs across the whole SmallEnsTrans ensemble: per-member metadata and aggregate
budgets (§2a), parameter-effect plots (§2b), ensemble vs observations (§2c), and
Sobol sensitivity indices / functional ANOVA (§2d).

Shared setup lives in `elmer_analysis.py`. **Single-member vs observation
comparison is in `AnalyseMemberVsObs.ipynb`.**

## §0 — Configuration & setup

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import xugrid as xu
import matplotlib.pyplot as plt
import proplot as pplt              # figure styling used by the line plots
import itertools
from itertools import combinations as iter_combinations
from pathlib import Path
# Requires in the env: xugrid, xarray, proplot, cftime (for 360_day obs calendars).

In [ ]:
import elmer_analysis as ea

# ── EDIT ME: point this at YOUR ensemble ────────────────────────────────────
CONFIG = ea.Config(
    postpro_dir   = "/home/jagereli/Postdoc/Data/postpro/elmerugrid",

    member_kind   = "smallenstrans",
    member_dir    = "/media/jagereli/Expansion1/TLC_ISMIP7_ANT/SmallEnsTrans/RUN3/",
    member_glob   = "*/*t_aa_trans*.nc",
    member_glmelt = "fmp",                     # BMB treatment at the GL: 'fmp' or 'nmp'
    ens_agg_file  = "../DATA/ensemble_basin_aggregates3.nc",  # per-member per-catchment budget

    # Mesh-specific products for the ENSEMBLE mesh (different node/face count
    # than the OCX mesh -> different files).
    basins_file   = "../DATA/basins_mouginotGrid.nc",
    obs_mesh_file = "../DATA/obs_on_elmer_mesh_pv.nc",

    obs_ismip_file     = "../DATA/AntarcticaObsISMIP7-v1.2.nc",
    obs_discharge_file = "../DATA/AIS_discharge_BMHF14.nc",
)

globals().update(ea.init(CONFIG))

In [ ]:
# Observed grounding-line discharge (BMHF14 / IMBIE), used by §2c.
obs_disch = load_obs_discharge()
ENS_AGG_FILE = CONFIG.ens_agg_file

## §2 — Ensemble by characteristics

Compare members grouped by their design parameters, quantify parameter
sensitivity (Sobol), and show the ensemble envelope against observations.

### 2a — Member metadata & per-member aggregate

`meta` parses each member's design parameters from its folder name (SmallEnsTrans
convention — ensemble-only, so it lives here, not in §0).

`ensemble_basin_aggregates3.nc` holds `(time, simu, catchment)` diagnostics for
every member. It already exists — **run the build cell only to regenerate it**
(loops all members, slow).

In [ ]:
# Cell 2b — Parse member metadata from folder names  (FIXED)
import re
import pandas as pd

FOLDER_RE = re.compile(
    r"TRANSIENT(\d+)(nemo|sat)(fmp|nmp)(budd|rc)(cst|jag)(c_cut|c)$"
)

def parse_folder(folder_name: str) -> dict:
    m = FOLDER_RE.match(folder_name)
    if m is None:
        raise ValueError(f"Could not parse folder name: {folder_name!r}")
    inv, ocean, gl_melt, fric_law, n_hyp, c_ext = m.groups()
    return {
        "inv":      int(inv),        # inversion member (3, 38 or 39), NOT the transient simu index
        "ocean":    ocean,
        "gl_melt":  gl_melt,
        "fric_law": fric_law,
        "n_hyp":    n_hyp,
        "c_ext":    c_ext,
        "folder":   folder_name,
    }

# f.parent.name gives the folder (e.g. TRANSIENT38nemofmpbuddcstc)
# f.name gives the .nc file inside (e.g. restart_aa_trans_41.nc)
meta = pd.DataFrame([parse_folder(f.parent.name) for f in member_files])
meta.index.name = "simu_idx"
display(meta)

In [ ]:
# LOAD the saved aggregate (fast). Skip the build cell below unless regenerating.
ds = xr.open_dataset(ENS_AGG_FILE)
years = ds["time"].values
ds["smb"] = ds["smb_floating"] + ds["smb_grounded"]
ds["bmb"] = ds["bmb_floating"] + ds["bmb_grounded"]
print(ds.sizes)

In [ ]:
# === REGENERATE the aggregate (slow: loops every member). Normally skip. ===

# %% Cell 7 — UPDATED main loop: pass gl_melt per member
n_members = len(member_files)

keys = [
    "iaf_mass", "grounded_mass", "shelf_mass",
    "grounded_area", "floating_area",
    "smb_grounded", "smb_floating",
    "bmb_grounded", "bmb_floating",
    "calving_face", "calving_edge",
    "gl_discharge",
]
all_data = {k: np.full((n_time, n_members, n_basins), np.nan) for k in keys}

for m, (fpath, gl_melt_m) in enumerate(zip(member_files,
                                             meta["gl_melt"].values)):
    print(f"[{m+1:3d}/{n_members}]  {fpath.name}  (gl_melt={gl_melt_m})",
          flush=True)
    uds  = xu.open_dataset(str(fpath), mask_and_scale=True)
    diag = compute_member_diagnostics(uds, gl_melt=gl_melt_m)
    uds.close()
    for k in keys:
        all_data[k][:, m, :] = diag[k]

print("Done.")

# %% Cell 8 — UPDATED output netCDF with both calving methods

years = np.linspace(1990, 2014, 25)

def make_rate(arr): return np.gradient(arr, 1.0, axis=0)

attrs = {
    "iaf_mass":      ("Ice above flotation mass",                    "Gt"),
    "grounded_mass": ("Total grounded ice mass",                     "Gt"),
    "shelf_mass":    ("Total ice shelf mass",                        "Gt"),
    "grounded_area": ("Grounded ice area",                           "km2"),
    "floating_area": ("Floating ice area",                           "km2"),
    "smb_grounded":  ("SMB over grounded area",                      "Gt yr-1"),
    "smb_floating":  ("SMB over floating area",                      "Gt yr-1"),
    "bmb_grounded":  ("BMB over grounded area",                      "Gt yr-1"),
    "bmb_floating":  ("BMB over floating area",                      "Gt yr-1"),
    "calving_face":  ("Calving flux — face method (calving_front_flux × area)", "Gt yr-1"),
    "calving_edge":  ("Calving flux — edge method (v·n × h × L)",    "Gt yr-1"),
    "gl_discharge":  ("Ice discharge at grounding line",             "Gt yr-1"),
    "iaf_rate":      ("IAF mass change rate",                        "Gt yr-1"),
    "grounded_rate": ("Grounded mass change rate",                   "Gt yr-1"),
    "shelf_rate":    ("Ice shelf mass change rate",                  "Gt yr-1"),
}

dvars = {}
for k in keys:
    long, unit = attrs[k]
    dvars[k] = xr.DataArray(
        all_data[k], dims=["time", "simu", "catchment"],
        attrs={"long_name": long, "units": unit},
    )

for src, rk in [("iaf_mass",      "iaf_rate"),
                ("grounded_mass", "grounded_rate"),
                ("shelf_mass",    "shelf_rate")]:
    long, unit = attrs[rk]
    dvars[rk] = xr.DataArray(
        make_rate(all_data[src]), dims=["time", "simu", "catchment"],
        attrs={"long_name": long, "units": unit},
    )

ds_out = xr.Dataset(
    dvars,
    coords={
        "time":      xr.DataArray(years, dims=["time"],
                                  attrs={"long_name": "Year", "units": "yr"}),
        "simu":      xr.DataArray(np.arange(n_members), dims=["simu"]),
        "catchment": xr.DataArray(basin_ids, dims=["catchment"],
                                  attrs={"long_name": "Mouginot basin ID"}),
        "folder":    ("simu", meta["folder"].values),
        "ocean":     ("simu", meta["ocean"].values),
        "gl_melt":   ("simu", meta["gl_melt"].values),
        "fric_law":  ("simu", meta["fric_law"].values),
        "n_hyp":     ("simu", meta["n_hyp"].values),
        "c_ext":     ("simu", meta["c_ext"].values),
        "inv":       ("simu", meta["inv"].values),
    },
    attrs={
        "title":        "Elmer/Ice ensemble — Mouginot basin diagnostics",
        "rho_ice":      f"{RHO_ICE} kg m-3",
        "rho_sw":       f"{RHO_SW} kg m-3",
        "area_source":  "Computed from mesh geometry (shoelace formula)",
        "calving_note": ("Two calving methods saved: "
                         "calving_face = calving_front_flux*area (XIOS variable); "
                         "calving_edge = edge-flux integral at floating boundary"),
    },
)

ds_out.to_netcdf("ensemble_basin_aggregates3.nc", mode="w")
print(ds_out)

### 2b — Parameter labels & effect plots

In [ ]:
# Parameter / variable labels & colour cycle (ds already loaded in 2a).
PARAMS = ["inv", "ocean", "gl_melt", "fric_law", "n_hyp", "c_ext"]

PARAM_LABELS = {
    "inv":      "Inversion member",
    "ocean":    "Ocean forcing",
    "gl_melt":  "GL melt parametrisation",
    "fric_law": "Friction law",
    "n_hyp":    "Effective pressure",
    "c_ext":    "Friction extension",
}
VAR_LABELS = {
    "iaf_mass":     ("IAF mass",             "Gt"),
    "iaf_rate":     ("IAF mass change rate", "Gt yr⁻¹"),
    "smb":          ("SMB",                  "Gt yr⁻¹"),
    "bmb":          ("BMB",                  "Gt yr⁻¹"),
    "calving":      ("calving",              "Gt yr⁻¹"),
    "gl_discharge": ("GL discharge",         "Gt yr⁻¹"),
}
CYCLE = pplt.Cycle("tab10")

In [ ]:
# %% Cell P2 — Core plotting function
def plot_parameter_effect(
    catchment,
    params,
    variable: str = "iaf_mass",
    ax=None,
    title = None,
):
    """
    Plot sub-ensemble median ± min/max for a given catchment and parameter grouping.
 
    Parameters
    ----------
    catchment : int or 'all'
        Mouginot basin ID, or 'all' to sum over all catchments.
    params : str or list of str
        One or more of: 'inv', 'ocean', 'gl_melt', 'fric_law', 'n_hyp', 'c_ext'.
        All unique combinations of the chosen parameters define the sub-ensembles.
    variable : str
        One of 'iaf_mass', 'iaf_rate', 'smb', 'gl_discharge'.
    ax : proplot Axes, optional
        If None a new figure is created.
    title : str, optional
        Override the auto-generated title.
 
    Returns
    -------
    fig, ax  (or just ax if an existing axes was passed in)
    """
    if isinstance(params, str):
        params = [params]
 
    # ── Select variable & catchment ──────────────────────────────────────────
    da = ds[variable]                              # (time, simu, catchment)
 
    if catchment == "all":
        data = da.sum("catchment")                 # (time, simu)
        catch_label = "all catchments"
    else:
        data = da.sel(catchment=catchment)         # (time, simu)
        catch_label = f"catchment {catchment}"
 
    # data.values : (n_time, n_simu)
    data_np = data.values
 
    # ── Build sub-ensemble groups ────────────────────────────────────────────
    # Retrieve the coordinate arrays for the chosen params (length = n_simu)
    coord_arrays = {p: ds[p].values for p in params}
 
    # All unique combinations
    unique_vals   = [np.unique(coord_arrays[p]) for p in params]
    combinations  = list(itertools.product(*unique_vals))
 
    # ── Create figure if needed ───────────────────────────────────────────────
    standalone = ax is None
    if standalone:
        fig, ax = pplt.subplots(figsize=(7, 4))
    else:
        fig = ax.figure
 
    colors = [c["color"] for c in itertools.islice(CYCLE, len(combinations))]
 
    for combo, color in zip(combinations, colors):
        # Boolean mask over simu dimension
        mask = np.ones(ds.sizes["simu"], dtype=bool)
        for p, v in zip(params, combo):
            mask &= (coord_arrays[p] == v)
 
        if mask.sum() == 0:
            continue                               # combination not present
 
        sub = data_np[:, mask]                     # (n_time, n_members_in_group)
 
        median = np.nanmedian(sub, axis=1)
        lo     = np.nanmin   (sub, axis=1)
        hi     = np.nanmax   (sub, axis=1)
 
        label = " / ".join(str(v) for v in combo)
 
        ax.plot(years, median, color=color, label=label, lw=1.8)
        ax.fill_between(years, lo, hi, color=color, alpha=0.18)
 
    # ── Formatting ───────────────────────────────────────────────────────────
    var_long, var_unit = VAR_LABELS[variable]
    param_str = " × ".join(PARAM_LABELS.get(p, p) for p in params)
 
    ax.set_xlabel("Year")
    ax.set_ylabel(f"{var_long} ({var_unit})")
    ax.legend(loc="best", ncols=1, fontsize=7, title=param_str, titlefontsize=8)
    ax.format(xminorlocator=1)
 
    if standalone:
        fig.savefig(
            f"{variable}_catch{catchment}_{'_x_'.join(params)}.png",
            dpi=150, bbox_inches="tight",
        )
        return fig, ax
    return ax

In [ ]:
# %% Cell P2b — Core plotting function with observations
def plot_parameter_effect_obs(
    catchment,
    params,
    variable: str = "iaf_mass",
    ax=None,
    title = None,
    ref_year: int = 2000,
):
    """
    Like plot_parameter_effect, but overlays observations as a black dashed line.

    Obs definition by variable:
      iaf_mass     : cum. integral of (SMB_gr - D_obs), anchored to the model
                     ensemble-median IAF mass at ref_year (so both are total mass)
      iaf_rate     : annual SMB_gr - D_obs
      gl_discharge : D_obs (IMBIE) directly
      smb          : no obs
    """
    if isinstance(params, str):
        params = [params]

    # ── Select variable & catchment ──────────────────────────────────────────
    da = ds[variable]
    if catchment == "all":
        data = da.sum("catchment")
        catch_label = "all catchments"
        basin_indices = [int(cid) - 1 for cid in obs_catchment_ids]
    else:
        data = da.sel(catchment=catchment)
        catch_label = f"catchment {catchment}"
        basin_indices = [int(catchment) - 1]

    data_np = data.values   # (n_time, n_simu)

    # ── Helper: observed discharge interpolated & summed over basins ──────────
    def _obs_discharge():
        d_tot = np.zeros(len(years))
        any_valid = False
        for i_basin in basin_indices:
            d_obs_b = d_imbie[:, i_basin]
            valid   = ~np.isnan(d_obs_b)
            if valid.sum() < 2:
                continue
            any_valid = True
            d_tot += np.interp(
                years, obs_years_sel[valid], d_obs_b[valid],
                left=np.nan, right=np.nan,
            )
        return d_tot if any_valid else None

    # ── Build observed series ─────────────────────────────────────────────────
    obs_plot  = None
    obs_label = None

    if variable in ("iaf_mass", "iaf_rate"):
        smb_gr = ds["smb_grounded"]
        if catchment == "all":
            smb_np = smb_gr.sum("catchment").values.mean(axis=1)
        else:
            smb_np = smb_gr.sel(catchment=catchment).values.mean(axis=1)

        d_obs_total = _obs_discharge()
        if d_obs_total is not None:
            mb_obs_annual = smb_np - d_obs_total            # Gt yr⁻¹

            if variable == "iaf_mass":
                ref_idx = np.where(years == ref_year)[0][0]
                cum = np.nancumsum(mb_obs_annual)
                cum -= cum[ref_idx]                         # change relative to ref
                # anchor to model total mass at ref_year → both on total-mass scale
                model_ref = np.nanmedian(data_np[ref_idx, :])
                obs_plot  = model_ref + cum
                obs_label = f"SMB_gr − D_obs (anchored {ref_year})"
            else:  # iaf_rate
                obs_plot  = mb_obs_annual
                obs_label = "SMB_gr − D_obs"

    elif variable == "gl_discharge":
        obs_plot  = _obs_discharge()
        obs_label = "D_obs (IMBIE)"
    # variable == "smb": no obs counterpart, obs_plot stays None

    # ── Create figure if needed ───────────────────────────────────────────────
    standalone = ax is None
    if standalone:
        fig, ax = pplt.subplots(figsize=(7, 4))
    else:
        fig = ax.figure

    # ── Sub-ensemble groups ───────────────────────────────────────────────────
    coord_arrays = {p: ds[p].values for p in params}
    unique_vals  = [np.unique(coord_arrays[p]) for p in params]
    combinations = list(itertools.product(*unique_vals))
    colors = [c["color"] for c in itertools.islice(CYCLE, len(combinations))]

    for combo, color in zip(combinations, colors):
        mask = np.ones(ds.sizes["simu"], dtype=bool)
        for p, v in zip(params, combo):
            mask &= (coord_arrays[p] == v)
        if mask.sum() == 0:
            continue

        sub    = data_np[:, mask]
        median = np.nanmedian(sub, axis=1)
        lo     = np.nanmin   (sub, axis=1)
        hi     = np.nanmax   (sub, axis=1)
        label  = " / ".join(str(v) for v in combo)

        ax.plot(years, median, color=color, label=label, lw=1.8)
        ax.fill_between(years, lo, hi, color=color, alpha=0.18)

    # ── Observations overlay ──────────────────────────────────────────────────
    if obs_plot is not None:
        ax.plot(years, obs_plot, color="k", lw=1.8, linestyle="--",
                label=obs_label, zorder=10)

    # ── Formatting ───────────────────────────────────────────────────────────
    var_long, var_unit = VAR_LABELS[variable]
    param_str = " × ".join(PARAM_LABELS.get(p, p) for p in params)

    ax.set_xlabel("Year")
    ax.set_ylabel(f"{var_long} ({var_unit})")
    ax.legend(loc="best", ncols=1, fontsize=7, title=param_str, titlefontsize=8)
    ax.format(xminorlocator=1)

    if title is not None:
        ax.set_title(title, fontsize=9)

    if standalone:
        fig.savefig(
            f"{variable}_obs_catch{catchment}_{'_x_'.join(params)}.png",
            dpi=150, bbox_inches="tight",
        )
        return fig, ax
    return ax

In [ ]:
# %% Cell P3 — Convenience wrapper: panel figure for multiple variables
def plot_multi_variable(
    catchment,
    params,
    variables: list = ("iaf_mass", "iaf_rate", "smb", "gl_discharge"),
    ncols: int = 2,
):
    """
    Same as plot_parameter_effect but tiled over several variables.
    """
    if isinstance(params, str):
        params = [params]
 
    nrows = int(np.ceil(len(variables) / ncols))
    fig, axes = pplt.subplots(nrows=nrows, ncols=ncols,
                               figsize=(6 * ncols, 4 * nrows),
                               sharey=False)
 
    for ax, var in zip(axes, variables):
        plot_parameter_effect(catchment, params, variable=var, ax=ax)
 
    # Hide unused panels
    for ax in axes[len(variables):]:
        ax.set_visible(False)
 
    param_str = " × ".join(PARAM_LABELS.get(p, p) for p in params)
    if catchment == "all":
        catch_label = "all catchments"
    else:
        catch_label = f"catchment {catchment}"
 
    fig.suptitle(f"{catch_label}  —  grouped by {param_str}", fontsize=11, y=1.01)
    fig.savefig(
        f"multi_var_catch{catchment}_{'_x_'.join(params)}.png",
        dpi=150, bbox_inches="tight",
    )
    return fig, axes

In [ ]:
# %% Cell P5 — Parameter matrix plot
def plot_parameter_matrix(
    catchment,
    variable: str = "iaf_mass",
    params: list = PARAMS,
):
    """
    Matrix of sub-ensemble plots.
    Diagonal    : single parameter grouping
    Lower left  : two-parameter combination
    Upper right : empty
    """
    n = len(params)
    fig, axes = pplt.subplots(
        nrows=n, ncols=n,
        figsize=(3.5 * n, 3 * n),
        sharey=False, sharex=True,
        hspace=0.5, wspace=0.4,
    )

    var_long, var_unit = VAR_LABELS[variable]
    catch_label = "all catchments" if catchment == "all" else f"catchment {catchment}"

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]

            if j > i:
                # Upper triangle — hide
                ax.set_visible(False)

            elif i == j:
                # Diagonal — single parameter
                plot_parameter_effect(
                    catchment, params[i], variable=variable, ax=ax,
                    title=PARAM_LABELS.get(params[i], params[i]),
                )

            else:
                # Lower triangle — two parameters
                plot_parameter_effect(
                    catchment, [params[j], params[i]], variable=variable, ax=ax,
                    title=f"{PARAM_LABELS.get(params[j], params[j])}\n× {PARAM_LABELS.get(params[i], params[i])}",
                )

            # Only show y-label on leftmost column
            if j > 0 and ax.get_visible():
                ax.set_ylabel("")

            # Only show x-label on bottom row
            if i < n - 1 and ax.get_visible():
                ax.set_xlabel("")

    fig.suptitle(
        f"{var_long} — {catch_label}",
        fontsize=13, y=1.01,
    )
    fig.savefig(
        f"matrix_{variable}_catch{catchment}.png",
        dpi=150, bbox_inches="tight",
    )
    return fig, axes

In [ ]:
# ── Example parameter-effect calls ──────────────────────────────────────────
plot_parameter_effect("all", "ocean",   variable="gl_discharge")
plot_parameter_effect(11,    ["ocean", "gl_melt"], variable="iaf_mass")
plot_multi_variable("all", "fric_law")

### 2c — Ensemble vs observations

The observed-discharge overlay (`plot_parameter_effect_obs`) needs the BMHF14
arrays. Build them here (reusing the Part-1 loader) so the ensemble envelope can
be drawn against obs.

In [ ]:
# Expose obs discharge in the shape plot_parameter_effect_obs expects.
if obs_disch is not None:
    obs_years_sel, d_imbie, de_imbie = obs_disch
    obs_catchment_ids = np.arange(1, 19)
    plot_parameter_effect_obs("all", "ocean", variable="gl_discharge")
else:
    print("BMHF14 discharge obs not available -> obs overlay skipped.")

### 2d — Sobol sensitivity indices (functional ANOVA)

In [ ]:
# %% Cell S1 — Core Sobol computation (pure functional ANOVA)
from itertools import combinations as iter_combinations
import pandas as pd
 
def _closed_variance(data_t, pvals, subset):
    """
    V[E[Y | X_subset]]  —  variance of conditional means for a parameter subset.
    Handles unbalanced group sizes (inv has 3 levels, others have 2).
 
    Parameters
    ----------
    data_t : (n_simu,) float array — output values at one time step
    pvals  : dict  param_name → (n_simu,) array of parameter values
    subset : list of param names
    """
    n          = len(data_t)
    grand_mean = np.mean(data_t)
 
    combos = list(itertools.product(*[np.unique(pvals[p]) for p in subset]))
    V = 0.0
    for combo in combos:
        mask = np.ones(n, dtype=bool)
        for p, v in zip(subset, combo):
            mask &= (pvals[p] == v)
        if mask.sum() == 0:
            continue
        p_k   = mask.sum() / n          # group weight (handles unbalanced design)
        mu_k  = np.mean(data_t[mask])
        V    += p_k * mu_k ** 2
 
    return V - grand_mean ** 2
 
 
def compute_sobol(data_t, pvals, max_order=3):
    """
    Compute all Sobol indices up to `max_order` for one time step.
 
    Parameters
    ----------
    data_t    : (n_simu,) array
    pvals     : dict  param_name → (n_simu,) array
    max_order : int, 1–3
 
    Returns
    -------
    S : dict  tuple_of_param_names → Sobol index (float)
        e.g. {('inv',): 0.12,  ('inv','ocean'): 0.03, ...}
    total_var : float
    """
    params    = list(pvals.keys())
    total_var = np.var(data_t)
 
    if total_var < 1e-30:
        return {}, 0.0
 
    # Step 1: closed variances V[E[Y|X_u]] for every subset up to max_order
    V_closed = {}
    for order in range(1, max_order + 1):
        for subset in iter_combinations(params, order):
            V_closed[subset] = _closed_variance(data_t, pvals, list(subset))
 
    # Step 2: pure interaction variances via inclusion-exclusion
    V_pure = {}
    for order in range(1, max_order + 1):
        for subset in iter_combinations(params, order):
            V_pure[subset] = V_closed[subset]
            for sub_order in range(1, order):
                for sub in iter_combinations(subset, sub_order):
                    V_pure[subset] -= V_pure[sub]
 
    S = {subset: max(0.0, V_pure[subset]) / total_var for subset in V_pure}
    return S, total_var

In [ ]:
# %% Cell S2 — Compute Sobol indices for all time steps
def build_sobol_timeseries(
    catchment,
    variable: str  = "iaf_mass",
    params:   list = PARAMS,
    max_order: int = 3,
):
    """
    Returns a DataFrame with columns = parameter subsets (as strings),
    rows = time steps.
    """
    da   = ds[variable]
    data = da.sum("catchment") if catchment == "all" else da.sel(catchment=catchment)
    data_np = data.values                              # (n_time, n_simu)
 
    pvals = {p: ds[p].values for p in params}
 
    records = []
    for t in range(len(years)):
        S, _ = compute_sobol(data_np[t], pvals, max_order=max_order)
        row = {" × ".join(k): v for k, v in S.items()}
        records.append(row)
 
    df = pd.DataFrame(records, index=years)
    df.index.name = "year"
    return df

In [ ]:
# %% Cell S3 — Plotting function
def plot_sobol_timeseries(
    sobol_df: pd.DataFrame,
    catchment,
    variable: str = "iaf_mass",
    top_n_interactions: int = 6,
    stacked: bool = True,
):
    """
    Three-panel figure: 1st order (stacked area or lines),
                        2nd order (top_n lines),
                        3rd order (top_n lines).
 
    Parameters
    ----------
    stacked            : if True, 1st-order panel is a stacked area (shows
                         how much total variance is explained).
    top_n_interactions : how many 2nd/3rd order indices to label individually;
                         the rest are grouped as 'others'.
    """
    s1 = sobol_df[[c for c in sobol_df if c.count("×") == 0]]
    s2 = sobol_df[[c for c in sobol_df if c.count("×") == 1]]
    s3 = sobol_df[[c for c in sobol_df if c.count("×") == 2]]
 
    var_long, var_unit = VAR_LABELS[variable]
    catch_label = "all catchments" if catchment == "all" else f"catchment {catchment}"
 
    fig, axes = pplt.subplots(nrows=3, ncols=1,
                               figsize=(8, 10),
                               sharey=False, hspace=0.6)
    ax1, ax2, ax3 = axes
 
    # ── Friendly labels for single parameters ────────────────────────────────
    def friendly(col):
        parts = [p.strip() for p in col.split("×")]
        return " × ".join(PARAM_LABELS.get(p, p) for p in parts)
 
    colors10 = [c["color"] for c in itertools.islice(CYCLE, max(len(s1.columns), top_n_interactions))]
 
    # ── Panel 1: 1st-order Sobol indices ────────────────────────────────────
    labels1 = [friendly(c) for c in s1.columns]
 
    if stacked:
        ax1.stackplot(
            years,
            [s1[c].values for c in s1.columns],
            labels=labels1,
            colors=colors10[:len(s1.columns)],
            alpha=0.85,
        )
        # Overlay total explained (sum of all orders) as dashed line
        total_explained = sobol_df.sum(axis=1)
        ax1.plot(years, total_explained.values, color="k", lw=1.2,
                 linestyle="--", label="Total explained", zorder=5)
    else:
        for col, label, color in zip(s1.columns, labels1, colors10):
            ax1.plot(years, s1[col].values, label=label, color=color, lw=2)
 
    ax1.set_ylabel("Sobol index  S_i")
    ax1.set_title("1st order — main effects")
    ax1.legend(loc="upper left", ncols=2, fontsize=8)
    ax1.format(yformatter="%.2f", xminorlocator=1, ylim=(0, None))
 
    # ── Panel 2: 2nd-order (top N by mean value) ────────────────────────────
    mean_s2   = s2.mean().sort_values(ascending=False)
    top_s2    = mean_s2.head(top_n_interactions).index.tolist()
    other_s2  = mean_s2.tail(len(mean_s2) - top_n_interactions).index.tolist()
 
    for i, col in enumerate(top_s2):
        ax2.plot(years, s2[col].values,
                 label=friendly(col),
                 color=colors10[i], lw=2)
 
    if other_s2:
        ax2.plot(years, s2[other_s2].sum(axis=1).values,
                 color="gray", lw=1.2, linestyle=":", label=f"others ({len(other_s2)})")
 
    ax2.set_ylabel("Sobol index  S_ij")
    ax2.set_title(f"2nd order — pairwise interactions  (top {top_n_interactions} shown)")
    ax2.legend(loc="upper left", ncols=2, fontsize=7)
    ax2.format(yformatter="%.3f", xminorlocator=1, ylim=(0, None))
 
    # ── Panel 3: 3rd-order (top N) ────────────────────────────────────────
    mean_s3  = s3.mean().sort_values(ascending=False)
    top_s3   = mean_s3.head(top_n_interactions).index.tolist()
    other_s3 = mean_s3.tail(len(mean_s3) - top_n_interactions).index.tolist()
 
    for i, col in enumerate(top_s3):
        ax3.plot(years, s3[col].values,
                 label=friendly(col),
                 color=colors10[i], lw=2)
 
    if other_s3:
        ax3.plot(years, s3[other_s3].sum(axis=1).values,
                 color="gray", lw=1.2, linestyle=":", label=f"others ({len(other_s3)})")
 
    ax3.set_ylabel("Sobol index  S_ijk")
    ax3.set_xlabel("Year")
    ax3.set_title(f"3rd order — triple interactions  (top {top_n_interactions} shown)")
    ax3.legend(loc="upper left", ncols=2, fontsize=7)
    ax3.format(yformatter="%.4f", xminorlocator=1, ylim=(0, None))
 
    fig.suptitle(
        f"Sobol sensitivity indices — {var_long}\n{catch_label}",
        fontsize=12, y=1.01,
    )
    fig.savefig(f"sobol_{variable}_catch{catchment}.png",
                dpi=150, bbox_inches="tight")
    return fig, axes

In [ ]:
# %% Cell S4 — Heatmap of time-mean Sobol indices (all orders at a glance)
def plot_sobol_heatmap(
    sobol_df: pd.DataFrame,
    catchment,
    variable: str = "iaf_mass",
    year = None,
):
    """
    Symmetric matrix heatmap of mean Sobol indices.
    Diagonal = 1st order.
    Off-diagonal (i,j) = 2nd order S_ij.
    A separate panel below shows the top 3rd-order triplets as a bar chart.
 
    Parameters
    ----------
    year : if given, show indices at that specific year rather than the time mean.
    """
    params   = PARAMS
    n        = len(params)
    labels   = [PARAM_LABELS.get(p, p) for p in params]
 
    s1 = sobol_df[[c for c in sobol_df if c.count("×") == 0]]
    s2 = sobol_df[[c for c in sobol_df if c.count("×") == 1]]
    s3 = sobol_df[[c for c in sobol_df if c.count("×") == 2]]
 
    if year is not None:
        t_idx  = int(np.argmin(np.abs(years - year)))
        s1_val = s1.iloc[t_idx]
        s2_val = s2.iloc[t_idx]
        s3_val = s3.iloc[t_idx]
        suffix = f" (year {int(year)})"
    else:
        s1_val = s1.mean()
        s2_val = s2.mean()
        s3_val = s3.mean()
        suffix = " (time mean)"
 
    # Build matrix
    mat = np.zeros((n, n))
    for i, p in enumerate(params):
        mat[i, i] = s1_val.get(p, 0.0)
 
    for col in s2.columns:
        p1, p2 = [c.strip() for c in col.split("×")]
        if p1 in params and p2 in params:
            i, j = params.index(p1), params.index(p2)
            mat[i, j] = mat[j, i] = s2_val.get(col, 0.0)
 
    # Figure: 2 rows — heatmap + 3rd-order bar
    fig, axes = pplt.subplots(
        nrows=2, ncols=1,
        figsize=(12, 9),
        hspace=0.5,
        height_ratios=(3, 2),
    )
    ax_heat, ax_bar = axes
 
    im = ax_heat.pcolormesh(mat, cmap="Reds", vmin=0)
    ax_heat.set_xticks(np.arange(n) + 0.5)
    ax_heat.set_yticks(np.arange(n) + 0.5)
    ax_heat.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax_heat.set_yticklabels(labels, fontsize=8)
    ax_heat.set_title("1st order (diagonal) & 2nd order (off-diagonal)")
    fig.colorbar(im, ax=ax_heat, label="Sobol index")
 
    # Annotate cells
    for i in range(n):
        for j in range(n):
            ax_heat.text(j + 0.5, i + 0.5, f"{mat[i,j]:.3f}",
                         ha="center", va="center", fontsize=7,
                         color="white" if mat[i,j] > 0.5 * mat.max() else "k")
 
    # 3rd-order bar chart
    s3_sorted = s3_val.sort_values(ascending=False)
    bar_labels = [" × ".join(PARAM_LABELS.get(p.strip(), p.strip())
                              for p in col.split("×"))
                  for col in s3_sorted.index]
    colors = [c["color"] for c in itertools.islice(CYCLE, len(s3_sorted))]
    ax_bar.barh(bar_labels[::-1], s3_sorted.values[::-1], color=colors[::-1])
    ax_bar.set_xlabel("Sobol index  S_ijk")
    ax_bar.set_title("3rd order — triple interactions")
 
    var_long, _ = VAR_LABELS[variable]
    catch_label  = "all catchments" if catchment == "all" else f"catchment {catchment}"
    fig.suptitle(f"Sobol indices — {var_long}\n{catch_label}{suffix}",
                 fontsize=11, y=1.01)
    fig.savefig(f"sobol_heatmap_{variable}_catch{catchment}.png",
                dpi=150, bbox_inches="tight")
    return fig, axes

In [ ]:
# ── Example Sobol calls ─────────────────────────────────────────────────────
sobol_df = build_sobol_timeseries(catchment=11, variable="iaf_mass", max_order=3)
plot_sobol_timeseries(sobol_df, catchment=11, variable="iaf_mass")
plot_sobol_heatmap(sobol_df, catchment=11, variable="iaf_mass")